# USAGE: PUT RAW DATA IN ../data/wifi_samples.csv, PLUG-AND-PLAY

## Preliminary processing and cleaning of raw samples

In [6]:
# import json
import math
from io import BytesIO
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.collections import PolyCollection

CSV_PATH = Path("../data/wifi_samples.csv")
WDUTIL_RSSI_PLACEHOLDER_FILL_DBM = None
COORDINATE_PRIORITY = ("phone_gps", "mac_corelocation", "corelocation")
COORDINATE_LABELS = {
    "phone_gps": "phone GPS",
    "mac_corelocation": "mac CoreLocation",
    "corelocation": "legacy CoreLocation",
}
DATAFRAME_COLUMNS = [
    "measurement_set_id",
    "sample_index",
    "collector_id",
    "environment",
    "building",
    "floor",
    "waypoint_id",
    "measurement_set_timestamp_utc",
    "location_timestamp",
    "latitude",
    "longitude",
    "h_accuracy_m",
    "mac_location_timestamp",
    "mac_latitude",
    "mac_longitude",
    "mac_h_accuracy_m",
    "phone_location_timestamp",
    "phone_latitude",
    "phone_longitude",
    "phone_altitude_m",
    "selected_location_source",
    "selected_location_timestamp",
    "selected_latitude",
    "selected_longitude",
    "wdutil_rssi_dbm",
    "wdutil_rssi_effective_dbm",
    "wdutil_noise_dbm",
    "wdutil_noise_effective_dbm",
    "wdutil_tx_rate",
    "wdutil_zero_placeholder",
]
NUMERIC_COLUMNS = [
    "sample_index",
    "floor",
    "latitude",
    "longitude",
    "h_accuracy_m",
    "mac_latitude",
    "mac_longitude",
    "mac_h_accuracy_m",
    "phone_latitude",
    "phone_longitude",
    "phone_altitude_m",
    "wdutil_rssi_dbm",
    "wdutil_noise_dbm",
]

def resolve_rssi_placeholder_fill_value(wifi_df):
    '''
    NOTE USED: fill with np.nan for now
    '''
    
    if WDUTIL_RSSI_PLACEHOLDER_FILL_DBM is not None:
        return float(WDUTIL_RSSI_PLACEHOLDER_FILL_DBM)
    observed_rssi = wifi_df.loc[
        wifi_df["wdutil_rssi_dbm"].notna() & wifi_df["wdutil_rssi_dbm"].ne(0),
        "wdutil_rssi_dbm",
    ]
    if observed_rssi.empty:
        raise ValueError("Could not infer an RSSI fill value because there are no nonzero wdutil_rssi_dbm values.")

    observed_noise = wifi_df.loc[
        wifi_df["wdutil_noise_dbm"].notna() & wifi_df["wdutil_noise_dbm"].ne(0),
        "wdutil_noise_dbm",
    ]
    return float(observed_rssi.min()), float(observed_noise.min())

def summarize_coordinate_sources(wifi_df):
    counts = wifi_df["selected_location_source"].value_counts(dropna=False)
    parts = []
    for source in COORDINATE_PRIORITY:
        count = int(counts.get(source, 0))
        if count:
            parts.append(f"{count} {COORDINATE_LABELS[source]}")
    missing = int(wifi_df["selected_location_source"].isna().sum())
    if missing:
        parts.append(f"{missing} missing coordinates")
    return ", ".join(parts)

def build_wifi_dataframe(csv_path=CSV_PATH):
    wifi_df = pd.read_csv(csv_path, na_values=["null", "none", "nan"], keep_default_na=True)

    for column in wifi_df.columns:
        if pd.api.types.is_object_dtype(wifi_df[column]) or pd.api.types.is_string_dtype(wifi_df[column]):
            wifi_df[column] = wifi_df[column].map(
                lambda value: value.strip() if isinstance(value, str) else value
            )
    wifi_df = wifi_df.replace(r"^\\s*$", pd.NA, regex=True)

    for column in NUMERIC_COLUMNS:
        if column in wifi_df.columns:
            wifi_df[column] = pd.to_numeric(wifi_df[column], errors="coerce")

    tx_rate_mbps = pd.to_numeric(
        wifi_df["wdutil_tx_rate"].astype("string").str.extract(r"([0-9.]+)", expand=False),
        errors="coerce",
    )
    wifi_df["wdutil_zero_placeholder"] = (wifi_df["wdutil_rssi_dbm"].eq(0) | wifi_df["wdutil_noise_dbm"].eq(0))
    # (
    #     wifi_df["wdutil_rssi_dbm"].ge(0)
    #     & wifi_df["wdutil_noise_dbm"].ge(0)
    #     & tx_rate_mbps.fillna(0).eq(0)
    # )
    # rssi_fill_value, noise_fill_value = resolve_rssi_placeholder_fill_value(wifi_df)
    wifi_df["wdutil_rssi_effective_dbm"] = wifi_df["wdutil_rssi_dbm"]
    wifi_df["wdutil_noise_effective_dbm"] = wifi_df["wdutil_noise_dbm"]
    wifi_df.loc[wifi_df["wdutil_zero_placeholder"], "wdutil_rssi_effective_dbm"] = np.nan
    wifi_df.loc[wifi_df["wdutil_zero_placeholder"], "wdutil_noise_effective_dbm"] = np.nan
    # wifi_df.attrs["wdutil_rssi_placeholder_fill_dbm"] = rssi_fill_value

    phone_available = wifi_df["phone_latitude"].notna() & wifi_df["phone_longitude"].notna()
    mac_available = wifi_df["mac_latitude"].notna() & wifi_df["mac_longitude"].notna()
    core_available = wifi_df["latitude"].notna() & wifi_df["longitude"].notna()

    wifi_df["selected_location_source"] = pd.Series(pd.NA, index=wifi_df.index, dtype="object")
    wifi_df.loc[phone_available, "selected_location_source"] = "phone_gps"
    wifi_df.loc[~phone_available & mac_available, "selected_location_source"] = "mac_corelocation"
    wifi_df.loc[
        ~phone_available & ~mac_available & core_available,
        "selected_location_source",
    ] = "corelocation"

    wifi_df["selected_location_timestamp"] = (
        wifi_df["phone_location_timestamp"]
        .combine_first(wifi_df["mac_location_timestamp"])
        .combine_first(wifi_df["location_timestamp"])
    )
    wifi_df["selected_latitude"] = (
        wifi_df["phone_latitude"]
        .combine_first(wifi_df["mac_latitude"])
        .combine_first(wifi_df["latitude"])
    )
    wifi_df["selected_longitude"] = (
        wifi_df["phone_longitude"]
        .combine_first(wifi_df["mac_longitude"])
        .combine_first(wifi_df["longitude"])
    )

    return wifi_df.reindex(columns=DATAFRAME_COLUMNS)


def write_wifi_dataframe(wifi_df, output_path):
    output_path = Path(output_path)
    wifi_df.to_csv(output_path, index=False, na_rep="null")
    return output_path

## Save full raw data

In [8]:
wifi_df = build_wifi_dataframe()
print(f"Loaded {len(wifi_df)} rows into pandas.DataFrame with {wifi_df.shape[1]} columns.")
print("Coordinate priority for selected_location_* columns: phone GPS -> mac CoreLocation -> legacy CoreLocation")
print(
    f"Using NaN as the fill value for {int(wifi_df['wdutil_zero_placeholder'].sum())} wdutil zero-placeholder rows."
)
print(f"Selected coordinate coverage: {summarize_coordinate_sources(wifi_df)}.")

write_wifi_dataframe(wifi_df, '../data/data.csv')
display(wifi_df.head(12))

Loaded 978 rows into pandas.DataFrame with 30 columns.
Coordinate priority for selected_location_* columns: phone GPS -> mac CoreLocation -> legacy CoreLocation
Using NaN as the fill value for 156 wdutil zero-placeholder rows.
Selected coordinate coverage: 582 phone GPS, 273 mac CoreLocation, 96 legacy CoreLocation, 27 missing coordinates.


,measurement_set_id,sample_index,collector_id,environment,building,floor,waypoint_id,measurement_set_timestamp_utc,location_timestamp,latitude,...,selected_location_source,selected_location_timestamp,selected_latitude,selected_longitude,wdutil_rssi_dbm,wdutil_rssi_effective_dbm,wdutil_noise_dbm,wdutil_noise_effective_dbm,wdutil_tx_rate,wdutil_zero_placeholder
0,c5c32855-1b84-45d9-86c6-20c26f40a728,0,kevin,indoor,evgr d,3,evgr d kevin room,2026-04-10T03:37:57.793334+00:00,2026-04-10 03:37:57 +0000,37.427721,...,corelocation,2026-04-10 03:37:57 +0000,37.427721,-122.155860,-58,-58.0,-98,-98.0,173.0 Mbps,False
1,c5c32855-1b84-45d9-86c6-20c26f40a728,1,kevin,indoor,evgr d,3,evgr d kevin room,2026-04-10T03:37:57.793334+00:00,2026-04-10 03:37:57 +0000,37.427721,...,corelocation,2026-04-10 03:37:57 +0000,37.427721,-122.155860,-57,-57.0,-98,-98.0,173.0 Mbps,False
2,c5c32855-1b84-45d9-86c6-20c26f40a728,2,kevin,indoor,evgr d,3,evgr d kevin room,2026-04-10T03:37:57.793334+00:00,2026-04-10 03:37:57 +0000,37.427721,...,corelocation,2026-04-10 03:37:57 +0000,37.427721,-122.155860,-57,-57.0,-98,-98.0,173.0 Mbps,False
3,e88c9b05-6135-4ce1-a9fd-bfc79a0bda2d,0,kevin,indoor,evgr d,3,NaN,2026-04-10T03:39:47.036843+00:00,2026-04-10 03:39:47 +0000,37.427721,...,corelocation,2026-04-10 03:39:47 +0000,37.427721,-122.155860,-58,-58.0,-98,-98.0,173.0 Mbps,False
4,e88c9b05-6135-4ce1-a9fd-bfc79a0bda2d,1,kevin,indoor,evgr d,3,NaN,2026-04-10T03:39:47.036843+00:00,2026-04-10 03:39:47 +0000,37.427721,...,corelocation,2026-04-10 03:39:47 +0000,37.427721,-122.155860,-58,-58.0,-98,-98.0,173.0 Mbps,False
5,e88c9b05-6135-4ce1-a9fd-bfc79a0bda2d,2,kevin,indoor,evgr d,3,NaN,2026-04-10T03:39:47.036843+00:00,2026-04-10 03:39:47 +0000,37.427721,...,corelocation,2026-04-10 03:39:47 +0000,37.427721,-122.155860,-58,-58.0,-98,-98.0,173.0 Mbps,False
6,821e7931-81e7-4b20-b0ee-2425f04eb925,0,kevin,indoor,evgr d,3,sequoia hall fyo,2026-04-10T19:29:18.700934+00:00,2026-04-10 19:29:18 +0000,37.428897,...,corelocation,2026-04-10 19:29:18 +0000,37.428897,-122.172035,-45,-45.0,-96,-96.0,573.0 Mbps,False
7,821e7931-81e7-4b20-b0ee-2425f04eb925,1,kevin,indoor,evgr d,3,sequoia hall fyo,2026-04-10T19:29:18.700934+00:00,2026-04-10 19:29:18 +0000,37.428897,...,corelocation,2026-04-10 19:29:18 +0000,37.428897,-122.172035,-44,-44.0,-96,-96.0,573.0 Mbps,False
8,821e7931-81e7-4b20-b0ee-2425f04eb925,2,kevin,indoor,evgr d,3,sequoia hall fyo,2026-04-10T19:29:18.700934+00:00,2026-04-10 19:29:18 +0000,37.428897,...,corelocation,2026-04-10 19:29:18 +0000,37.428897,-122.172035,-44,-44.0,-96,-96.0,573.0 Mbps,False
9,b5760ea5-84ff-4ea9-a370-8306d5ab65f3,0,kevin,indoor,sequoia,2,sequoia fyo,2026-04-10T19:31:58.942286+00:00,2026-04-10 19:31:58 +0000,37.428971,...,corelocation,2026-04-10 19:31:58 +0000,37.428971,-122.172236,-44,-44.0,-96,-96.0,573.0 Mbps,False


## Clean data to prepare for inference

In [9]:
# grab "interesting" columns, drop anything without location
df_clean = wifi_df[['environment', 'floor',
                    'selected_location_timestamp',
                    'selected_latitude',
                    'selected_longitude',
                    'wdutil_rssi_effective_dbm',
                    'wdutil_noise_effective_dbm',
                    'wdutil_zero_placeholder']].copy()
df_clean = df_clean.dropna(subset=['selected_latitude', 'selected_longitude'])
df_clean['wdutil_zero_placeholder'] = ~df_clean['wdutil_zero_placeholder']
df_clean.columns = ['environment', 'floor', 'datetime', 'latitude', 'longitude', 'rssi', 'noise', 'signal_measured']

df_clean['indoor'] = (df_clean['environment'] == 'indoor')

from datetime import timezone, timedelta

# convert to datetime and all to PDT
df_clean['datetime'] = pd.to_datetime(df_clean['datetime'], format='mixed', utc=True).dt.tz_convert("America/Los_Angeles")
df_clean['date'] = df_clean['datetime'].dt.date
df_clean['time_pdt'] = df_clean['datetime'].dt.time

# reorder columns
df_clean = df_clean[['date', 'time_pdt', 'latitude', 'longitude', 'indoor', 'floor', 'rssi', 'noise', 'signal_measured', 'datetime']]


write_wifi_dataframe(df_clean, '../data/df_clean.csv')
display(df_clean.head(12))

,date,time_pdt,latitude,longitude,indoor,floor,rssi,noise,signal_measured,datetime
0,2026-04-09,20:37:57,37.427721,-122.155860,True,3,-58.0,-98.0,True,2026-04-09 20:37:57-07:00
1,2026-04-09,20:37:57,37.427721,-122.155860,True,3,-57.0,-98.0,True,2026-04-09 20:37:57-07:00
2,2026-04-09,20:37:57,37.427721,-122.155860,True,3,-57.0,-98.0,True,2026-04-09 20:37:57-07:00
3,2026-04-09,20:39:47,37.427721,-122.155860,True,3,-58.0,-98.0,True,2026-04-09 20:39:47-07:00
4,2026-04-09,20:39:47,37.427721,-122.155860,True,3,-58.0,-98.0,True,2026-04-09 20:39:47-07:00
5,2026-04-09,20:39:47,37.427721,-122.155860,True,3,-58.0,-98.0,True,2026-04-09 20:39:47-07:00
6,2026-04-10,12:29:18,37.428897,-122.172035,True,3,-45.0,-96.0,True,2026-04-10 12:29:18-07:00
7,2026-04-10,12:29:18,37.428897,-122.172035,True,3,-44.0,-96.0,True,2026-04-10 12:29:18-07:00
8,2026-04-10,12:29:18,37.428897,-122.172035,True,3,-44.0,-96.0,True,2026-04-10 12:29:18-07:00
9,2026-04-10,12:31:58,37.428971,-122.172236,True,2,-44.0,-96.0,True,2026-04-10 12:31:58-07:00
